# Data evaluation

In [13]:
import os
import json

In [14]:
experiments = [f"experiment_{i}" for i in range(1,18)]
DATA_PATH = "../data/"
ID_MAP_PATH = './movie_ids_map.json'
BASENAME_PATH = "./similar_movies.json"

THRESHOLD = 0.1
SAMPLE_SIZE = 100

In [15]:
def get_movie_ids_map():
    with open(ID_MAP_PATH, 'r') as f:
        movie_ids_map = json.load(f)
        
    return {int(k): v for k,v in movie_ids_map.items()}

In [16]:
def from_dict_of_lists_to_tuples(d):
    return [(int(k), v) for k, values in d.items() for v in values]

def get_baseline_data():
    with open(BASENAME_PATH, 'r') as f:
        baseline_similar_items = json.load(f)
    return baseline_similar_items

In [17]:
baseline_results = from_dict_of_lists_to_tuples(get_baseline_data())

In [18]:
def get_results_from_file(filename):
    with open(filename, 'r') as f:
        tuples_list = [eval(line.strip()) for line in f]
    return tuples_list

def get_parameters_from_file(filename):
    with open(filename, 'r') as f:
        parameters = json.load(f)
    return parameters

In [22]:
def get_pairs_over_threshold(l,t):
    filtered_tuples = filter(lambda r: r[1]>=t, l)
    return list(map(lambda r: r[0], list(filtered_tuples)))

def compute_confusion_matrix(baseline, experiment, n):
    baseline_set = set(baseline) 
    experiment_set = set(experiment)
    
    tp = len(baseline_set & experiment_set)  
    fp = len(experiment_set - baseline_set)  
    fn = len(baseline_set - experiment_set)  
    tn = n - tp - fp - fn
    
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def compute_sensitivity_specificity(tp, fn, fp, tn):
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    return sensitivity, specificity

In [ ]:
results_with_value = {}
parameters = {}
results_mapped = {}
for exp in experiments:
    filename = os.path.join(DATA_PATH,f'results_{exp}')
    results_with_value[exp] = get_results_from_file(filename)
    filename = os.path.join(DATA_PATH,f'parameters_{exp}.json')
    parameters[exp] = get_parameters_from_file(filename)
    id_map = get_movie_ids_map()
    results_mapped[exp] = list(map(lambda r: ((id_map[r[0][0]], id_map[r[0][1]]),r[1]), results_with_value[exp]))
    results_mapped[exp] += [((r[0][1], r[0][0]), r[1]) for r in results_mapped[exp]] #consider both orders of pairs
    if THRESHOLD:
        results_mapped[exp] = get_pairs_over_threshold(results_mapped[exp], THRESHOLD)

Experiment:  experiment_1
Parameters:  {'t_PCA': 0.95, 'l': 100, 't': 0.8, 'b': 10, 'h': 2048}
{'TP': 4, 'FP': 12, 'FN': 2268, 'TN': 7616}
(0.0017605633802816902, 0.9984268484530676)

Experiment:  experiment_2
Parameters:  {'t_PCA': 0.95, 'l': 200, 't': 0.8, 'b': 17, 'h': 8192}
{'TP': 0, 'FP': 4, 'FN': 2272, 'TN': 7624}
(0.0, 0.9994756161510225)

Experiment:  experiment_3
Parameters:  {'t_PCA': 0.95, 'l': 100, 't': 0.7, 'b': 10, 'h': 2048}
{'TP': 9, 'FP': 21, 'FN': 2263, 'TN': 7607}
(0.003961267605633803, 0.9972469847928683)

Experiment:  experiment_4
Parameters:  {'t_PCA': 0.95, 'l': 100, 't': 0.9, 'b': 10, 'h': 2048}
{'TP': 0, 'FP': 0, 'FN': 2272, 'TN': 7628}
(0.0, 1.0)

Experiment:  experiment_5
Parameters:  {'t_PCA': 0.95, 'l': 200, 't': 0.7, 'b': 17, 'h': 8192}
{'TP': 9, 'FP': 15, 'FN': 2263, 'TN': 7613}
(0.003961267605633803, 0.9980335605663345)

Experiment:  experiment_6
Parameters:  {'t_PCA': 0.95, 'l': 200, 't': 0.9, 'b': 17, 'h': 8192}
{'TP': 1, 'FP': 1, 'FN': 2271, 'TN': 762

In [ ]:
for exp in experiments:
    total_pairs = SAMPLE_SIZE*(SAMPLE_SIZE-1)
    conf = compute_confusion_matrix(baseline_results, results_mapped[exp], total_pairs)
    print("Experiment: ", exp)
    print("Parameters: ", parameters[exp])
    print(conf)
    print(compute_sensitivity_specificity(conf["TP"], conf["FN"], conf["FP"], conf["TN"]))
    print()